In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print (device)

cuda


In [ ]:
from torchvision import transforms

transform_augmented = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(32, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),

    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [ ]:
train_dataset=torchvision.datasets.CIFAR10(root="/data",train=True,download=True,transform=transform_augmented)
test_dataset=torchvision.datasets.CIFAR10(root="/data",train=False,download=True,transform=transform_augmented)


In [ ]:
print(len(train_dataset))
print(len(test_dataset))

50000
10000


In [ ]:
train_size=int(.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset,val_dataset = random_split(train_dataset,[train_size,val_size])

In [ ]:
batch_size= 64
train_loader= DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader= DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader= DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class SimpleCNN(nn.Module):
  def __init__(self):
    super(SimpleCNN,self).__init__()

    self.conv1 = nn.Conv2d(3,32,3,padding=1)
    self.conv2 = nn.Conv2d(32,64,3,padding=1)

    self.pool=nn.MaxPool2d(2,2)

    self.fc1=nn.Linear(64*8*8,256)
    self.fc2=nn.Linear(256,10)
    self.relu = nn.ReLU()

  def forward(self,x):
    x=self.pool(self.relu(self.conv1(x)))
    x=self.pool(self.relu(self.conv2(x)))

    x=x.view(x.size(0),-1)
    x=self.relu(self.fc1(x))
    x=self.fc2(x)
    return x



In [ ]:
model = SimpleCNN().to(device)
loss_f = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=.001)

In [ ]:
def train_model(model, train_loader):
  model.train()
  total_loss= 0
  for images,labels in train_loader:
    images,labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss= loss_f(outputs,labels)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  return total_loss/len(train_loader)



In [ ]:
def evaluate(model,test_loader):
  model.eval()
  correct=0
  total=0

  with torch.no_grad():
    for images,labels in test_loader:
      images,labels = images.to(device), labels.to(device)
      outputs = model(images)

      _,predicted = torch.max(outputs,1)

      total += labels.size(0)
      correct += (predicted==labels).sum().item()

  return 100*correct/total


In [ ]:
epochs=5
for e in range(epochs):
  train_loss = train_model(model,train_loader)
  val_acc = evaluate(model, val_loader)
  print(f"Epoch {e+1}: Train Loss = {train_loss:.4f}, Val Acc = {val_acc:.2f}%")




Epoch 1: Train Loss = 1.5051, Val Acc = 54.01%
Epoch 2: Train Loss = 1.2043, Val Acc = 60.22%
Epoch 3: Train Loss = 1.0814, Val Acc = 62.40%
Epoch 4: Train Loss = 1.0004, Val Acc = 64.23%
Epoch 5: Train Loss = 0.9410, Val Acc = 66.74%
